In [0]:
%sql

CREATE OR REPLACE TABLE job_market.silver.hn_posts_unified AS

WITH combined AS (

    SELECT
        post_id,
        thread_id,
        posted_year,
        
CASE
    WHEN CAST(posted_month AS STRING)
         RLIKE '^[0-9]{4}-[0-9]{2}$'
    THEN TRY_CAST(
        SUBSTRING(CAST(posted_month AS STRING), 6, 2)
        AS INT
    )
    ELSE TRY_CAST(posted_month AS INT)
END AS posted_month,

        skills,
        is_remote,
        CAST(role_family AS STRING) AS role_family,
        CAST(region AS STRING) AS region,
        'cotera' AS data_source,
        1 AS source_priority

    FROM job_market.silver.hacker_news_posts

    UNION ALL

    SELECT
        post_id,
        thread_id,
        posted_year,
        TRY_CAST(posted_month AS INT) AS posted_month,
        skills,
        is_remote,
        CAST(NULL AS STRING) AS role_family,
        CAST(NULL AS STRING) AS region,
        'api' AS data_source,
        2 AS source_priority

    FROM job_market.silver.hacker_news_api_enriched_v2
),

deduplicated AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY post_id
            ORDER BY source_priority
        ) AS rn
    FROM combined
)

SELECT
    post_id,
    thread_id,
    posted_year,
    posted_month,
    skills,
    is_remote,
    role_family,
    region,
    data_source

FROM deduplicated
WHERE rn = 1;


In [0]:
%sql

SELECT
    data_source,
    COUNT(*) AS total_posts,
    COUNT_IF(posted_month IS NULL) AS null_months,
    COUNT_IF(posted_month NOT BETWEEN 1 AND 12)
        AS invalid_months
FROM job_market.silver.hn_posts_unified
GROUP BY data_source;


In [0]:
%sql

SELECT
    COUNT(*) AS total_posts,
    COUNT(DISTINCT post_id) AS unique_posts
FROM job_market.silver.hn_posts_unified;
